# Movies — Modèles NLP avancés

Ce notebook compare **trois approches plus avancées** pour la classification des critiques de films (`P/N`) :

1. **FastText**
2. **DistilBERT**
3. **BERT**

## Objectif
Comparer ces modèles à votre meilleure baseline classique :
- **TF-IDF + LinearSVC optimisé**

## Conseils
- Exécuter ce notebook sur **Google Colab**
- Pour DistilBERT et BERT, activer de préférence **GPU**
- Garder le notebook baseline et le notebook improvements séparés

## 1. Installation des dépendances

- `fasttext` pour le modèle FastText
- `transformers` et `datasets` pour DistilBERT / BERT
- `accelerate` pour faciliter l'entraînement avec Hugging Face

In [ ]:
# Si besoin sur Colab, décommente cette cellule
!pip install -q fasttext transformers datasets accelerate scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Imports

In [ ]:
from pathlib import Path
import os
import re
import string
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import matplotlib.pyplot as plt

## 3. Fixer la seed

In [5]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

## 4. Chargement des données

Adapte `DATA_DIR` à ton Drive.

Structure attendue :

```text
movies1000/
    pos/
    neg/
```

ou bien :

```text
movies1000/
    movies1000/
        pos/
        neg/
```

In [6]:
DATA_DIR = Path("/content/drive/MyDrive/projet tal/movies1000/movies1000")
print("DATA_DIR existe :", DATA_DIR.exists())
if DATA_DIR.exists():
    print("Contenu :", [p.name for p in DATA_DIR.iterdir()])

DATA_DIR existe : True
Contenu : ['neg', 'pos']


In [7]:
def load_movies_from_folder(data_dir: Path) -> pd.DataFrame:
    rows = []

    for label_folder, label in [("pos", "P"), ("neg", "N")]:
        folder = data_dir / label_folder
        if not folder.exists():
            continue

        for file_path in folder.glob("*.txt"):
            text = file_path.read_text(encoding="utf-8", errors="ignore")
            rows.append({
                "doc_id": file_path.name,
                "label": label,
                "text": text,
            })

    if not rows:
        raise ValueError("Aucun fichier trouvé. Vérifie DATA_DIR.")

    return pd.DataFrame(rows).sort_values("doc_id").reset_index(drop=True)

df = load_movies_from_folder(DATA_DIR)
print(df.shape)
df.head()

(2000, 3)


,doc_id,label,text
0,cv000_29416.txt,N,"plot : two teen couples go to a church party ,..."
1,cv000_29590.txt,P,films adapted from comic books have had plenty...
2,cv001_18431.txt,P,every now and then a movie comes along from a ...
3,cv001_19502.txt,N,the happy bastard's quick movie review \ndamn ...
4,cv002_15918.txt,P,you've got mail works alot better than it dese...


## 5. Vérification rapide

In [8]:
print(df["label"].value_counts().sort_index())
df["n_words"] = df["text"].str.split().str.len()
display(df.groupby("label")["n_words"].agg(["mean", "median", "std", "min", "max"]))

label
N    1000
P    1000
Name: count, dtype: int64


,mean,median,std,min,max
label,,,,,
N,705.630,668.5,296.729759,17,2181
P,787.051,731.5,352.799881,130,2678


## 6. Préprocessing modéré

On réutilise ici le préprocessing standard qui avait bien marché :
- minuscules
- suppression ponctuation
- suppression chiffres
- normalisation espaces

In [9]:
def preprocess_standard(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text_standard"] = df["text"].apply(preprocess_standard)
df[["text", "text_standard"]].head(3)

,text,text_standard
0,"plot : two teen couples go to a church party ,...",plot two teen couples go to a church party dri...
1,films adapted from comic books have had plenty...,films adapted from comic books have had plenty...
2,every now and then a movie comes along from a ...,every now and then a movie comes along from a ...


## 7. Split train / validation

In [10]:
train_df, valid_df = train_test_split(
    df[["doc_id", "label", "text", "text_standard"]].copy(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print("Train :", train_df.shape)
print("Valid :", valid_df.shape)

Train : (1600, 4)
Valid : (400, 4)


## 8. Fonction utilitaire d'évaluation

In [11]:
def evaluate_predictions(y_true, y_pred, positive_label="P"):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=positive_label),
        "recall": recall_score(y_true, y_pred, pos_label=positive_label),
        "f1": f1_score(y_true, y_pred, pos_label=positive_label),
    }
    return pd.DataFrame({"score": metrics})

def print_report(y_true, y_pred, title):
    print(f"\n===== {title} =====")
    print(classification_report(y_true, y_pred, digits=4))
    display(evaluate_predictions(y_true, y_pred))

##**version un fichier plusieur reviews**

In [12]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/content/drive/MyDrive/projet tal/movies1000/movies1000")

def load_movies_one_review_per_line(data_dir: Path) -> pd.DataFrame:
    rows = []

    for label_folder, label in [("pos", "P"), ("neg", "N")]:
        folder = data_dir / label_folder

        if not folder.exists():
            continue

        for file_path in folder.glob("*.txt"):
            content = file_path.read_text(encoding="utf-8", errors="ignore")

            # On considère qu'une review = une ligne non vide
            reviews = [line.strip() for line in content.splitlines() if line.strip()]

            for i, review in enumerate(reviews):
                rows.append({
                    "source_file": file_path.name,
                    "review_id": f"{file_path.stem}_{i}",
                    "label": label,
                    "text": review,
                })

    if not rows:
        raise ValueError("Aucune review trouvée. Vérifie DATA_DIR et la structure des fichiers.")

    df = pd.DataFrame(rows).reset_index(drop=True)
    return df

df = load_movies_one_review_per_line(DATA_DIR)

print(df.shape)
display(df.head(10))
print(df["label"].value_counts())

(64720, 4)


,source_file,review_id,label,text
0,cv570_29082.txt,cv570_29082_0,P,plot : a group of asbestos cleaners get a job ...
1,cv570_29082.txt,cv570_29082_1,P,"as each day passes , the crew members begin to..."
2,cv570_29082.txt,cv570_29082_2,P,saying anything else about the plot would be a...
3,cv570_29082.txt,cv570_29082_3,P,have fun . . .
4,cv570_29082.txt,cv570_29082_4,P,"critique : "" i feel like shooting myself in th..."
5,cv570_29082.txt,cv570_29082_5,P,"this ain't your average "" happy go lucky "" kin..."
6,cv570_29082.txt,cv570_29082_6,P,this is a deliberately slow-paced mystery-horr...
7,cv570_29082.txt,cv570_29082_7,P,will it bore some people to sleep ?
8,cv570_29082.txt,cv570_29082_8,P,you bet it will !
9,cv570_29082.txt,cv570_29082_9,P,is it made for the scream audiences of the day ?


label
P    32937
N    31783
Name: count, dtype: int64


In [13]:
import re
import string

def preprocess_standard(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text_standard"] = df["text"].apply(preprocess_standard)
display(df[["text", "text_standard"]].head(10))

,text,text_standard
0,plot : a group of asbestos cleaners get a job ...,plot a group of asbestos cleaners get a job re...
1,"as each day passes , the crew members begin to...",as each day passes the crew members begin to d...
2,saying anything else about the plot would be a...,saying anything else about the plot would be a...
3,have fun . . .,have fun
4,"critique : "" i feel like shooting myself in th...",critique i feel like shooting myself in the he...
5,"this ain't your average "" happy go lucky "" kin...",this aint your average happy go lucky kind of ...
6,this is a deliberately slow-paced mystery-horr...,this is a deliberately slowpaced mysteryhorror...
7,will it bore some people to sleep ?,will it bore some people to sleep
8,you bet it will !,you bet it will
9,is it made for the scream audiences of the day ?,is it made for the scream audiences of the day


In [14]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(
    df[["source_file", "review_id", "label", "text", "text_standard"]].copy(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print("Train :", train_df.shape)
print("Valid :", valid_df.shape)
print("\nRépartition train :")
print(train_df["label"].value_counts())
print("\nRépartition valid :")
print(valid_df["label"].value_counts())

Train : (51776, 5)
Valid : (12944, 5)

Répartition train :
label
P    26350
N    25426
Name: count, dtype: int64

Répartition valid :
label
P    6587
N    6357
Name: count, dtype: int64


# Partie A — FastText

FastText est un bon compromis :
- plus avancé que TF-IDF
- plus léger que BERT
- rapide à entraîner
- adapté à la classification de texte

## 9. Préparer les fichiers FastText

In [ ]:
def label_to_fasttext(label: str) -> str:
    return f"__label__{label}"

FASTTEXT_DIR = Path("/content/drive/MyDrive/projet tal/fasttext_tmp")
FASTTEXT_DIR.mkdir(parents=True, exist_ok=True)

fasttext_train_path = FASTTEXT_DIR / "movies_train_fasttext.txt"
fasttext_valid_path = FASTTEXT_DIR / "movies_valid_fasttext.txt"

def write_fasttext_file(dataframe: pd.DataFrame, output_path: Path, text_col: str = "text_standard"):
    with output_path.open("w", encoding="utf-8") as f:
        for _, row in dataframe.iterrows():
            label = label_to_fasttext(row["label"])
            text = str(row[text_col]).replace("\n", " ").strip()
            f.write(f"{label} {text}\n")

write_fasttext_file(train_df, fasttext_train_path, text_col="text_standard")
write_fasttext_file(valid_df, fasttext_valid_path, text_col="text_standard")

print("Train file :", fasttext_train_path)
print("Valid file :", fasttext_valid_path)

Train file : /content/drive/MyDrive/projet tal/fasttext_tmp/movies_train_fasttext.txt
Valid file : /content/drive/MyDrive/projet tal/fasttext_tmp/movies_valid_fasttext.txt


## 10. Entraîner FastText

In [ ]:
import fasttext

fasttext_model = fasttext.train_supervised(
    input=str(fasttext_train_path),
    lr=0.5,
    epoch=30,
    wordNgrams=2,
    dim=100,
    loss="softmax",
    thread=4,
)

fasttext_model

## 11. Évaluer FastText

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import numpy as np

texts_valid = valid_df["text_standard"].astype(str).tolist()
labels_pred, scores = fasttext_model.predict(texts_valid, k=1)
preds = [lab[0].replace("__label__", "") for lab in labels_pred]

print(classification_report(valid_df["label"], preds, digits=4))

metrics_valid = pd.DataFrame({
    "score": {
        "accuracy": accuracy_score(valid_df["label"], preds),
        "precision": precision_score(valid_df["label"], preds, pos_label="P"),
        "recall": recall_score(valid_df["label"], preds, pos_label="P"),
        "f1": f1_score(valid_df["label"], preds, pos_label="P"),
    }
})

display(metrics_valid)

              precision    recall  f1-score   support

           N     0.8808    0.8500    0.8651       200
           P     0.8551    0.8850    0.8698       200

    accuracy                         0.8675       400
   macro avg     0.8680    0.8675    0.8675       400
weighted avg     0.8680    0.8675    0.8675       400



,score
accuracy,0.867500
precision,0.855072
recall,0.885000
f1,0.869779


## 12. Petit tuning FastText

In [ ]:
fasttext_param_grid = [
    {"lr": 0.5, "epoch": 25, "wordNgrams": 2, "dim": 100},
    {"lr": 0.3, "epoch": 30, "wordNgrams": 2, "dim": 100},
    {"lr": 0.5, "epoch": 25, "wordNgrams": 3, "dim": 100},
    {"lr": 0.5, "epoch": 30, "wordNgrams": 2, "dim": 200},
]

fasttext_results = []

texts_valid = valid_df["text_standard"].astype(str).tolist()
y_true = valid_df["label"].tolist()

for params in fasttext_param_grid:
    model = fasttext.train_supervised(
        input=str(fasttext_train_path),
        lr=params["lr"],
        epoch=params["epoch"],
        wordNgrams=params["wordNgrams"],
        dim=params["dim"],
        loss="softmax",
        thread=4,
    )

    labels, scores = model.predict(texts_valid, k=1)
    preds = [lab[0].replace("__label__", "") for lab in labels]

    res = {
        **params,
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, pos_label="P"),
        "recall": recall_score(y_true, preds, pos_label="P"),
        "f1": f1_score(y_true, preds, pos_label="P"),
    }
    fasttext_results.append(res)

fasttext_results_df = (
    pd.DataFrame(fasttext_results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

fasttext_results_df

,lr,epoch,wordNgrams,dim,accuracy,precision,recall,f1
0,0.5,30,2,200,0.860,0.830275,0.905,0.866029
1,0.5,25,2,100,0.850,0.839806,0.865,0.852217
2,0.3,30,2,100,0.825,0.828283,0.820,0.824121
3,0.5,25,3,100,0.800,0.785714,0.825,0.804878


In [ ]:
from pathlib import Path
import pandas as pd
import fasttext

# 1) Charger le test
TEST_FILE = Path("/content/testSentiment (1).txt")

def load_test_one_review_per_line(test_file: Path) -> pd.DataFrame:
    raw_text = test_file.read_text(encoding="utf-8", errors="ignore")
    lines = raw_text.split("\n")

    print("Nombre total de lignes dans le fichier test :", len(lines))

    non_empty_lines = [line.strip() for line in lines if line.strip()]
    print("Nombre de lignes non vides (= samples utilisés) :", len(non_empty_lines))

    rows = []
    for i, line in enumerate(non_empty_lines):
        rows.append({
            "doc_id": i,
            "text": line
        })

    return pd.DataFrame(rows)

test_df = load_test_one_review_per_line(TEST_FILE)

print("\nShape du DataFrame test :", test_df.shape)
print("Nombre final de samples test :", len(test_df))

# 2) Préprocessing identique à l'entraînement
test_df["text_standard"] = test_df["text"].apply(preprocess_standard)


# 4) Prédictions sur le test
texts_test = test_df["text_standard"].astype(str).tolist()
labels, scores = fasttext_model.predict(texts_test, k=1)
test_pred_fasttext = [lab[0].replace("__label__", "") for lab in labels]

# 5) Création du fichier de soumission
submission_fasttext = pd.DataFrame({
    "label": test_pred_fasttext
})

print("\nNombre de prédictions générées :", len(submission_fasttext))
display(submission_fasttext.head(10))
print(submission_fasttext["label"].value_counts())

# 6) Sauvegarde CSV
output_path = "/content/drive/MyDrive/projet tal/submission-movies-fasttext-1.csv"
submission_fasttext.to_csv(output_path, index=False)

print(f"\nFichier enregistré : {output_path}")

Nombre total de lignes dans le fichier test : 25001
Nombre de lignes non vides (= samples utilisés) : 25000

Shape du DataFrame test : (25000, 2)
Nombre final de samples test : 25000

Nombre de prédictions générées : 25000


,label
0,N
1,P
2,N
3,N
4,N
5,N
6,N
7,P
8,P
9,N


label
N    14236
P    10764
Name: count, dtype: int64

Fichier enregistré : /content/drive/MyDrive/projet tal/submission-movies-fasttext-1.csv


In [ ]:
submission_fasttext.to_csv(output_path, index=False, header=False)

# Partie B — DistilBERT

DistilBERT est une version plus légère de BERT :
- plus rapide
- moins coûteuse
- souvent très performante

## 13. Imports transformers

In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

## 14. Préparer les données pour Hugging Face

In [ ]:
label2id = {"N": 0, "P": 1}
id2label = {0: "N", 1: "P"}

hf_train_df = train_df[["text", "label"]].copy()
hf_valid_df = valid_df[["text", "label"]].copy()

hf_train_df["label_id"] = hf_train_df["label"].map(label2id)
hf_valid_df["label_id"] = hf_valid_df["label"].map(label2id)

hf_train = Dataset.from_pandas(hf_train_df[["text", "label_id"]].rename(columns={"label_id": "label"}))
hf_valid = Dataset.from_pandas(hf_valid_df[["text", "label_id"]].rename(columns={"label_id": "label"}))

hf_train, hf_valid

(Dataset({
     features: ['text', 'label'],
     num_rows: 1600
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 400
 }))

## 15. Fonction de métriques pour transformers

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    y_true = np.array([id2label[int(x)] for x in labels])
    y_pred = np.array([id2label[int(x)] for x in preds])

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label="P"),
        "recall": recall_score(y_true, y_pred, pos_label="P"),
        "f1": f1_score(y_true, y_pred, pos_label="P"),
    }

## 16. DistilBERT — Tokenizer + tokenisation

In [ ]:
DISTILBERT_MODEL_NAME = "distilbert-base-uncased"

distilbert_tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_MODEL_NAME)

def tokenize_distilbert(batch):
    return distilbert_tokenizer(batch["text"], truncation=True)

tokenized_train_distil = hf_train.map(tokenize_distilbert, batched=True)
tokenized_valid_distil = hf_valid.map(tokenize_distilbert, batched=True)

data_collator_distil = DataCollatorWithPadding(tokenizer=distilbert_tokenizer)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

## 17. DistilBERT — Modèle

In [ ]:
distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 18. DistilBERT — Entraînement

In [ ]:
distil_output_dir = "/content/drive/MyDrive/projet tal/distilbert_movies_output"

distil_training_args = TrainingArguments(
    output_dir=distil_output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    save_total_limit=1,
)

In [ ]:
distil_trainer = Trainer(
    model=distilbert_model,
    args=distil_training_args,
    train_dataset=tokenized_train_distil,
    eval_dataset=tokenized_valid_distil,
    data_collator=data_collator_distil,
    compute_metrics=compute_metrics,
)

In [ ]:
distil_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.512451,0.518480,0.730000,0.659722,0.950000,0.778689
2,0.275247,0.424750,0.837500,0.842640,0.830000,0.836272


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=400, training_loss=0.39384902954101564, metrics={'train_runtime': 11508.5246, 'train_samples_per_second': 0.278, 'train_steps_per_second': 0.035, 'total_flos': 423895675699200.0, 'train_loss': 0.39384902954101564, 'epoch': 2.0})

## 19. DistilBERT — Évaluation

In [ ]:
# Décommente après distil_trainer.train()
distil_eval = distil_trainer.evaluate()
print(distil_eval)

{'eval_loss': 0.42474982142448425, 'eval_accuracy': 0.8375, 'eval_precision': 0.8426395939086294, 'eval_recall': 0.83, 'eval_f1': 0.836272040302267, 'eval_runtime': 340.1785, 'eval_samples_per_second': 1.176, 'eval_steps_per_second': 0.073, 'epoch': 2.0}


In [ ]:
# Décommente après entraînement pour récupérer les prédictions détaillées
distil_preds_output = distil_trainer.predict(tokenized_valid_distil)
distil_preds = np.argmax(distil_preds_output.predictions, axis=1)
distil_pred_labels = [id2label[int(x)] for x in distil_preds]

print_report(valid_df["label"], distil_pred_labels, "DistilBERT")

# Partie C — BERT

Même logique que DistilBERT, mais avec un modèle plus lourd.
Ici on utilise `bert-base-uncased`, cohérent avec des reviews en anglais.

## 20. BERT — Tokenizer + tokenisation

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
BERT_MODEL_NAME = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

def tokenize_bert(batch):
    return bert_tokenizer(batch["text"], truncation=True)

tokenized_train_bert = hf_train.map(tokenize_bert, batched=True)
tokenized_valid_bert = hf_valid.map(tokenize_bert, batched=True)

data_collator_bert = DataCollatorWithPadding(tokenizer=bert_tokenizer)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

## 21. BERT — Modèle

In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 22. BERT — Entraînement

In [ ]:
bert_output_dir = "/content/drive/MyDrive/projet tal/bert_movies_output"

bert_training_args = TrainingArguments(
    output_dir=bert_output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    save_total_limit=1,
)

In [ ]:
bert_trainer = Trainer(
    model=bert_model,
    args=bert_training_args,
    train_dataset=tokenized_train_bert,
    eval_dataset=tokenized_valid_bert,
    data_collator=data_collator_bert,
    compute_metrics=compute_metrics,
)

bert_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.511021,0.382751,0.820000,0.905063,0.715000,0.798883
2,0.262498,0.473765,0.847500,0.806167,0.915000,0.857143


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=400, training_loss=0.38675919532775876, metrics={'train_runtime': 362.9291, 'train_samples_per_second': 8.817, 'train_steps_per_second': 1.102, 'total_flos': 841955377152000.0, 'train_loss': 0.38675919532775876, 'epoch': 2.0})

## 23. BERT — Évaluation

In [ ]:
# Décommente après bert_trainer.train()
bert_eval = bert_trainer.evaluate()
print(bert_eval)

{'eval_loss': 0.47376549243927, 'eval_accuracy': 0.8475, 'eval_precision': 0.8061674008810573, 'eval_recall': 0.915, 'eval_f1': 0.8571428571428571, 'eval_runtime': 12.4909, 'eval_samples_per_second': 32.023, 'eval_steps_per_second': 2.001, 'epoch': 2.0}


In [ ]:
# Décommente après entraînement pour récupérer les prédictions détaillées
bert_preds_output = bert_trainer.predict(tokenized_valid_bert)
bert_preds = np.argmax(bert_preds_output.predictions, axis=1)
bert_pred_labels = [id2label[int(x)] for x in bert_preds]

print_report(valid_df["label"], bert_pred_labels, "BERT")


===== BERT =====
              precision    recall  f1-score   support

           N     0.9017    0.7800    0.8365       200
           P     0.8062    0.9150    0.8571       200

    accuracy                         0.8475       400
   macro avg     0.8540    0.8475    0.8468       400
weighted avg     0.8540    0.8475    0.8468       400



,score
accuracy,0.847500
precision,0.806167
recall,0.915000
f1,0.857143


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from datasets import Dataset

# 1) Charger le test
TEST_FILE = Path("/content/drive/MyDrive/projet tal/testSentiment.txt")

def load_test_one_review_per_line(test_file: Path) -> pd.DataFrame:
    raw_text = test_file.read_text(encoding="utf-8", errors="ignore")
    lines = raw_text.split("\n")

    print("Nombre total de lignes dans le fichier test :", len(lines))

    non_empty_lines = [line.strip() for line in lines if line.strip()]
    print("Nombre de lignes non vides (= samples utilisés) :", len(non_empty_lines))

    rows = []
    for i, line in enumerate(non_empty_lines):
        rows.append({
            "doc_id": i,
            "text": line
        })

    return pd.DataFrame(rows)

test_df = load_test_one_review_per_line(TEST_FILE)

print("\nShape du DataFrame test :", test_df.shape)
print("Nombre final de samples test :", len(test_df))

# 2) Créer le dataset Hugging Face
hf_test = Dataset.from_pandas(test_df[["text"]].copy())

# 3) Tokenisation BERT
def tokenize_test_bert(batch):
    return bert_tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
    )

tokenized_test_bert = hf_test.map(tokenize_test_bert, batched=True)

if "text" in tokenized_test_bert.column_names:
    tokenized_test_bert = tokenized_test_bert.remove_columns(["text"])

# 4) Prédictions avec BERT
bert_preds_output = bert_trainer.predict(tokenized_test_bert)
bert_pred_ids = np.argmax(bert_preds_output.predictions, axis=1)

# 5) Conversion vers labels P/N
test_pred_bert = [id2label[int(x)] for x in bert_pred_ids]

# 6) Création du fichier de soumission sans nom de colonne
submission_bert = pd.DataFrame(test_pred_bert)

print("\nNombre de prédictions générées :", len(submission_bert))
display(submission_bert.head(10))
print(submission_bert[0].value_counts())

# 7) Sauvegarde CSV sans header
output_path = "/content/drive/MyDrive/projet tal/submission-movies-bert-1.csv"
submission_bert.to_csv(output_path, index=False, header=False)

print(f"\nFichier enregistré : {output_path}")

Nombre total de lignes dans le fichier test : 25001
Nombre de lignes non vides (= samples utilisés) : 25000

Shape du DataFrame test : (25000, 2)
Nombre final de samples test : 25000


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]


Nombre de prédictions générées : 25000


,0
0,P
1,P
2,N
3,N
4,N
5,P
6,N
7,P
8,P
9,P


0
P    13789
N    11211
Name: count, dtype: int64

Fichier enregistré : /content/drive/MyDrive/projet tal/submission-movies-bert-1.csv


## 24. Tableau récapitulatif à remplir

In [ ]:
summary_rows = [
    {"model": "FastText", "accuracy": None, "precision": None, "recall": None, "f1": None},
    {"model": "DistilBERT", "accuracy": None, "precision": None, "recall": None, "f1": None},
    {"model": "BERT", "accuracy": None, "precision": None, "recall": None, "f1": None},
]

summary_df = pd.DataFrame(summary_rows)
summary_df

## 25. Modèle final retenu

À compléter après comparaison.

Exemple :
- si FastText est proche du meilleur SVM mais plus simple, tu peux le garder comme modèle NLP léger
- si DistilBERT surpasse clairement les autres, il devient le meilleur modèle avancé
- si BERT n'apporte pas de gain net par rapport à DistilBERT, DistilBERT peut rester le meilleur compromis

## 26. Prochaine étape

Quand tu auras exécuté ce notebook, montre-moi :
- les résultats `fasttext_results_df`
- les scores de DistilBERT
- les scores de BERT

et on décidera :
- quel modèle garder
- lequel mettre dans le notebook de soumission avancée
- comment l'expliquer dans le rapport

**test** roberta xlm

In [2]:
# Si besoin sur Colab, décommente :
!pip install -q transformers datasets accelerate scikit-learn

In [3]:
from pathlib import Path
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

In [4]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

In [5]:
DATA_DIR = Path("/content/drive/MyDrive/projet tal/movies1000/movies1000")
print("DATA_DIR existe :", DATA_DIR.exists())
if DATA_DIR.exists():
    print("Contenu :", [p.name for p in DATA_DIR.iterdir()])

DATA_DIR existe : True
Contenu : ['neg', 'pos']


In [6]:
def load_movies_from_folder(data_dir: Path) -> pd.DataFrame:
    rows = []

    for label_folder, label in [("pos", "P"), ("neg", "N")]:
        folder = data_dir / label_folder
        if not folder.exists():
            continue

        for file_path in folder.glob("*.txt"):
            text = file_path.read_text(encoding="utf-8", errors="ignore")
            rows.append({
                "doc_id": file_path.name,
                "label": label,
                "text": text,
            })

    if not rows:
        raise ValueError("Aucun fichier trouvé. Vérifie DATA_DIR.")

    return pd.DataFrame(rows).sort_values("doc_id").reset_index(drop=True)

df = load_movies_from_folder(DATA_DIR)
print(df.shape)
df.head()

(2000, 3)


,doc_id,label,text
0,cv000_29416.txt,N,"plot : two teen couples go to a church party ,..."
1,cv000_29590.txt,P,films adapted from comic books have had plenty...
2,cv001_18431.txt,P,every now and then a movie comes along from a ...
3,cv001_19502.txt,N,the happy bastard's quick movie review \ndamn ...
4,cv002_15918.txt,P,you've got mail works alot better than it dese...


In [7]:
train_df, valid_df = train_test_split(
    df[["doc_id", "label", "text"]].copy(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print("Train :", train_df.shape)
print("Valid :", valid_df.shape)
print(train_df["label"].value_counts().sort_index())

Train : (1600, 3)
Valid : (400, 3)
label
N    800
P    800
Name: count, dtype: int64


In [8]:
label2id = {"N": 0, "P": 1}
id2label = {0: "N", 1: "P"}

hf_train_df = train_df[["text", "label"]].copy()
hf_valid_df = valid_df[["text", "label"]].copy()

hf_train_df["label_id"] = hf_train_df["label"].map(label2id)
hf_valid_df["label_id"] = hf_valid_df["label"].map(label2id)

hf_train = Dataset.from_pandas(
    hf_train_df[["text", "label_id"]].rename(columns={"label_id": "label"})
)
hf_valid = Dataset.from_pandas(
    hf_valid_df[["text", "label_id"]].rename(columns={"label_id": "label"})
)

hf_train, hf_valid

(Dataset({
     features: ['text', 'label'],
     num_rows: 1600
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 400
 }))

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    y_true = np.array([id2label[int(x)] for x in labels])
    y_pred = np.array([id2label[int(x)] for x in preds])

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label="P"),
        "recall": recall_score(y_true, y_pred, pos_label="P"),
        "f1": f1_score(y_true, y_pred, pos_label="P"),
    }

def print_report(y_true, y_pred, title):
    print(f"\n===== {title} =====")
    print(classification_report(y_true, y_pred, digits=4))
    metrics = pd.DataFrame({
        "score": {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, pos_label="P"),
            "recall": recall_score(y_true, y_pred, pos_label="P"),
            "f1": f1_score(y_true, y_pred, pos_label="P"),
        }
    })
    display(metrics)

In [10]:
def run_transformer_experiment(
    model_name: str,
    run_name: str,
    hf_train,
    hf_valid,
    valid_df,
    num_epochs: int = 2,
    learning_rate: float = 2e-5,
    batch_size_train: int = 8,
    batch_size_eval: int = 16,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_fn(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=512,
        )

    tokenized_train = hf_train.map(tokenize_fn, batched=True)
    tokenized_valid = hf_valid.map(tokenize_fn, batched=True)

    if "text" in tokenized_train.column_names:
        tokenized_train = tokenized_train.remove_columns(["text"])
    if "text" in tokenized_valid.column_names:
        tokenized_valid = tokenized_valid.remove_columns(["text"])

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    output_dir = f"/content/drive/MyDrive/projet tal/{run_name}_output"

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size_train,
        per_device_eval_batch_size=batch_size_eval,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        report_to="none",
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_valid,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    train_output = trainer.train()
    eval_output = trainer.evaluate()

    pred_output = trainer.predict(tokenized_valid)
    pred_ids = np.argmax(pred_output.predictions, axis=1)
    pred_labels = [id2label[int(x)] for x in pred_ids]

    print_report(valid_df["label"], pred_labels, run_name)

    return {
        "tokenizer": tokenizer,
        "trainer": trainer,
        "train_output": train_output,
        "eval_output": eval_output,
        "pred_labels": pred_labels,
    }

## Partie B — CardiffNLP Twitter-XLM-RoBERTa

In [11]:
#TWITTER_XLMR_MODEL = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
TWITTER_ROBERTA_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"

In [23]:
'''# Décommente pour lancer l'expérience
twitter_xlmr_results = run_transformer_experiment(
     model_name=TWITTER_XLMR_MODEL,
     run_name="twitter_xlmr_cardiffnlp",
     hf_train=hf_train,
     hf_valid=hf_valid,
     valid_df=valid_df,
     num_epochs=2,
     learning_rate=2e-5,
     batch_size_train=8,
     batch_size_eval=16,
 )'''
twitter_roberta_results = run_transformer_experiment(
    model_name=TWITTER_ROBERTA_MODEL,
    run_name="twitter_roberta_cardiffnlp",
    hf_train=hf_train,
    hf_valid=hf_valid,
    valid_df=valid_df,
    num_epochs=2,
    learning_rate=2e-5,
    batch_size_train=8,
    batch_size_eval=16,
)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
roberta.embeddings.position_ids | UNEXPECTED |                                                                                     
roberta.pooler.dense.bias       | UNEXPECTED |                                                                                     
roberta.pooler.dense.weight     | UNEXPECTED |                                                                                     
classifier.out_proj.bias        | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.out_proj.weight      | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs mod

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.442873,0.315493,0.880000,0.836283,0.945000,0.887324
2,0.266414,0.392173,0.900000,0.912371,0.885000,0.898477


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


===== twitter_roberta_cardiffnlp =====
              precision    recall  f1-score   support

           N     0.8883    0.9150    0.9015       200
           P     0.9124    0.8850    0.8985       200

    accuracy                         0.9000       400
   macro avg     0.9004    0.9000    0.9000       400
weighted avg     0.9004    0.9000    0.9000       400



,score
accuracy,0.900000
precision,0.912371
recall,0.885000
f1,0.898477


In [14]:
TEST_FILE = Path("/content/drive/MyDrive/projet tal/testSentiment.txt")

def load_test_one_review_per_line(test_file: Path) -> pd.DataFrame:
    lines = test_file.read_text(encoding="utf-8", errors="ignore").split("\n")
    rows = []
    for i, line in enumerate(lines):
        line = line.strip()
        if line:
            rows.append({
                "doc_id": i,
                "text": line
            })
    return pd.DataFrame(rows)

test_df = load_test_one_review_per_line(TEST_FILE)
print(test_df.shape)
test_df.head()

(25000, 2)


,doc_id,text
0,0,Story of a man who has unnatural feelings for ...
1,1,Bromwell High is a cartoon comedy. It ran at t...
2,2,Airport '77 starts as a brand new luxury 747 p...
3,3,This film lacked something I couldn't put my f...
4,4,"Sorry everyone,,, I know this is supposed to b..."


In [26]:
# Décommente après avoir entraîné twitter_roberta_results
twitter_roberta_trainer = twitter_roberta_results["trainer"]
twitter_roberta_tokenizer = twitter_roberta_results["tokenizer"]

hf_test = Dataset.from_pandas(test_df[["text"]].copy())

def tokenize_test_twitter_roberta(batch):
    return twitter_roberta_tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
    )

tokenized_test = hf_test.map(tokenize_test_twitter_roberta, batched=True)
if "text" in tokenized_test.column_names:
     tokenized_test = tokenized_test.remove_columns(["text"])

test_preds_output = twitter_roberta_trainer.predict(tokenized_test)
test_pred_ids = np.argmax(test_preds_output.predictions, axis=1)
test_pred_labels = [id2label[int(x)] for x in test_pred_ids]

submission_twitter_roberta = pd.DataFrame({"label": test_pred_labels})
display(submission_twitter_roberta.head(10))

output_path = "/content/drive/MyDrive/projet tal/submission_movies_twitter_roberta.csv"
submission_twitter_roberta.to_csv(output_path, index=False)
print(f"Fichier enregistré : {output_path}")

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [12]:
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "/content/drive/MyDrive/projet tal/twitter_roberta_cardiffnlp_output/checkpoint-400"

# si tu ne connais pas le bon checkpoint, regarde le contenu du dossier parent
print(Path("/content/drive/MyDrive/projet tal/twitter_roberta_cardiffnlp_output").exists())

twitter_roberta_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

twitter_roberta_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
twitter_roberta_model.to(device)
twitter_roberta_model.eval()

print("Modèle rechargé depuis :", MODEL_DIR)

True


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modèle rechargé depuis : /content/drive/MyDrive/projet tal/twitter_roberta_cardiffnlp_output/checkpoint-400


In [15]:
import torch
import pandas as pd
from tqdm.auto import tqdm

texts_test = test_df["text"].astype(str).tolist()

def predict_in_batches(model, tokenizer, texts, batch_size=8, max_length=512):
    all_pred_ids = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]

        encodings = tokenizer(
            batch_texts,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors="pt",
        )

        encodings = {k: v.to(device) for k, v in encodings.items()}

        with torch.no_grad():
            outputs = model(**encodings)
            logits = outputs.logits
            pred_ids = torch.argmax(logits, dim=1).cpu().numpy().tolist()

        all_pred_ids.extend(pred_ids)

    return all_pred_ids

test_pred_ids = predict_in_batches(
    model=twitter_roberta_model,
    tokenizer=twitter_roberta_tokenizer,
    texts=texts_test,
    batch_size=8,
    max_length=512,
)

test_pred_labels = [id2label[int(x)] for x in test_pred_ids]

submission_twitter_roberta = pd.DataFrame(test_pred_labels)

print("Nombre de prédictions :", len(submission_twitter_roberta))
display(submission_twitter_roberta.head(10))
print(submission_twitter_roberta[0].value_counts())

output_path = "/content/drive/MyDrive/projet tal/submission_movies_twitter_roberta.csv"
submission_twitter_roberta.to_csv(output_path, index=False, header=False)

print(f"Fichier enregistré : {output_path}")

  0%|          | 0/3125 [00:00<?, ?it/s]

Nombre de prédictions : 25000


,0
0,P
1,P
2,N
3,N
4,N
5,P
6,N
7,P
8,P
9,P


0
P    13552
N    11448
Name: count, dtype: int64
Fichier enregistré : /content/drive/MyDrive/projet tal/submission_movies_twitter_roberta.csv
